In [1]:
from pathlib import Path

BASE = Path.cwd()
DATA = BASE / "data"
RAW = DATA / "raw"
INTERIM = DATA / "interim"

for p in [DATA, RAW, INTERIM]:
    p.mkdir(parents=True, exist_ok=True)

print("Folders created.")

Folders created.


In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
PAH_GENES = {"PAH"}
CFTR_GENES = {"CFTR"}
HEREDITARY_CANCER_GENES = {
    "BRCA1", "BRCA2", "TP53", "PALB2", "ATM", "CHEK2",
    "MLH1", "MSH2", "MSH6", "PMS2", "APC", "PTEN",
    "CDH1", "STK11", "RAD51C", "RAD51D", "BARD1", "BRIP1"
}

print("PAH genes:", PAH_GENES)
print("CFTR genes:", CFTR_GENES)
print("Hereditary cancer genes:", len(HEREDITARY_CANCER_GENES))

PAH genes: {'PAH'}
CFTR genes: {'CFTR'}
Hereditary cancer genes: 18


In [4]:
needed_clinvar_columns = [
    "VariationID",
    "GeneSymbol",
    "Type",
    "Assembly",
    "OriginSimple",
    "ClinicalSignificance",
    "ReviewStatus",
    "Chromosome",
    "Start",
    "ReferenceAllele",
    "AlternateAllele",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "Name"
]

In [5]:
clinvar_path = RAW / "variant_summary.txt.gz"

clinvar = pd.read_csv(
    clinvar_path,
    sep="\t",
    compression="gzip",
    low_memory=False,
)   
print(clinvar.shape)
clinvar.head(3)

(8907105, 43)


,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,ReferenceAllele,AlternateAllele,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
0,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTAACTGTAAA (p.Arg27_Ile28delinsLeuLeuTer),9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,-,RCV000000012|RCV005255549|RCV004998069,"MONDO:MONDO:0013342,MedGen:C3150901,OMIM:613647,Orphanet:306511||MedGen:C3661900",Hereditary spastic paraplegia 48|Macular dystrophy with or without extraocular features|not provided,germline;unknown,germline,GRCh37,NC_000007.13,7,4820844,4820847,na,na,7p22.1,"criteria provided, multiple submitters, no conflicts",4,-,N,"ClinGen:CA215070,OMIM:613653.0001",3,2,4820844,GGAT,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
1,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTAACTGTAAA (p.Arg27_Ile28delinsLeuLeuTer),9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,-,RCV000000012|RCV005255549|RCV004998069,"MONDO:MONDO:0013342,MedGen:C3150901,OMIM:613647,Orphanet:306511||MedGen:C3661900",Hereditary spastic paraplegia 48|Macular dystrophy with or without extraocular features|not provided,germline;unknown,germline,GRCh38,NC_000007.14,7,4781213,4781216,na,na,7p22.1,"criteria provided, multiple submitters, no conflicts",4,-,N,"ClinGen:CA215070,OMIM:613653.0001",3,2,4781213,GGAT,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
2,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,-,RCV000000013,"MONDO:MONDO:0013342,MedGen:C3150901,OMIM:613647,Orphanet:306511",Hereditary spastic paraplegia 48,germline,germline,GRCh37,NC_000007.13,7,4827361,4827374,na,na,7p22.1,no assertion criteria provided,1,-,N,"ClinGen:CA215072,OMIM:613653.0002",1,3,4827360,GCTGCTGGACCTGCC,G,-,-,-,-,-,-,SCV000020156,-,-


In [6]:
existing = [c for c in needed_clinvar_columns if c in clinvar.columns]
missing = [c for c in needed_clinvar_columns if c not in clinvar.columns]

print("Existing:", existing)
print("Missing:", missing)

Existing: ['VariationID', 'GeneSymbol', 'Type', 'Assembly', 'OriginSimple', 'ClinicalSignificance', 'ReviewStatus', 'Chromosome', 'Start', 'ReferenceAllele', 'AlternateAllele', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'Name']
Missing: []


In [8]:
clinvar_f = clinvar.copy()

# 1) Assembly
clinvar_f = clinvar_f[clinvar_f["Assembly"] == "GRCh38"]

# 2) SNV only
clinvar_f = clinvar_f[
    clinvar_f["Type"].str.contains("single nucleotide variant", case=False, na=False)
]

# 3) Germline only
origin_col = "OriginSimple" if "OriginSimple" in clinvar_f.columns else "Origin"
clinvar_f = clinvar_f[
    clinvar_f[origin_col].str.contains("germline", case=False, na=False)
]

# 4) Trusted review status only
trusted_statuses = {
    "reviewed by expert panel",
    "practice guideline"
}
clinvar_f = clinvar_f[
    clinvar_f["ReviewStatus"].isin(trusted_statuses)
]

print(clinvar_f.shape)
print(clinvar_f['ReviewStatus'].value_counts(dropna=False))
clinvar_f[["ClinicalSignificance", "ReviewStatus"]].head()

(14970, 43)
ReviewStatus
reviewed by expert panel    14923
practice guideline             47
Name: count, dtype: int64


,ClinicalSignificance,ReviewStatus
547,Pathogenic,reviewed by expert panel
549,Pathogenic,reviewed by expert panel
553,Likely pathogenic,reviewed by expert panel
555,Pathogenic,reviewed by expert panel
557,Pathogenic,reviewed by expert panel


In [10]:
def normalize_label(clinsig: str):
    if pd.isna(clinsig):
        return np.nan
    
    s = str(clinsig).lower()
    
    # Reject unclear or conflicting entries first
    bad_terms = ["conflicting", "uncertain", "risk factor", "association", "drug response"]
    if any(term in s for term in bad_terms):
        return np.nan

    has_path = ("pathogenic" in s)
    has_benign = ("benign" in s)

    # mixed pathogenic + benign records are not acceptable
    if has_path and has_benign:
        return np.nan

    if "likely pathogenic" in s or s == "pathogenic" or "pathogenic" in s:
        return "Pathogenic"
    if "likely benign" in s or s == "benign" or "benign" in s:
        return "Benign"

    return np.nan

clinvar_f["label"] = clinvar_f["ClinicalSignificance"].apply(normalize_label)
clinvar_f = clinvar_f.dropna(subset=["label"]).copy()

print(clinvar_f["label"].value_counts(dropna=False))
print(clinvar_f.shape)

label
Pathogenic    6140
Benign        5094
Name: count, dtype: int64
(11234, 44)


In [12]:
variant_key = ["Chromosome", "Start", "ReferenceAlleleVCF", "AlternateAlleleVCF"]

clinvar_f = clinvar_f.dropna(subset=variant_key).copy()
clinvar_f = clinvar_f.drop_duplicates(subset=variant_key).copy()

print(clinvar_f["label"].value_counts(dropna=False))
print(clinvar_f.shape)

label
Pathogenic    5419
Benign        4891
Name: count, dtype: int64
(10310, 44)


In [13]:
print(clinvar_f["GeneSymbol"].unique())

['VWF' 'PAH' 'APC' 'ACADVL' 'MSH2' 'ADA' 'ADA;PKIG' 'ANO5' 'VHL' 'ETHE1'
 'USH2A' 'ITGA2B' 'ATM' 'KCNQ1' 'LDLR' 'LDLR;MIR6886' 'GAA' 'GP1BA'
 'DICER1' 'DCLRE1C' 'SLC26A4' 'CDH23' 'AIPL1' 'OTOF' 'KCNQ4' 'COCH' 'DYSF'
 'SHOC2' 'MYO15A' 'CFTR' 'DNM2' 'GATM' 'TCF4' 'CYP1B1' 'PTEN' 'ABCA4'
 'MYOC' 'UBE3A' 'SGCD' 'ACVRL1' 'MAP2K2' 'GAMT' 'SGCB' 'FOXN1' 'BMPR2'
 'MSH6' 'HNF4A' 'PMS2' 'BRCA2' 'GUCY2D' 'JAK3' 'SGCA' 'RRAS2' 'MT-TV'
 'MT-TW' 'MT-TS2' 'MT-TS1' 'MT-CO1;MT-TS1' 'MT-TP' 'MT-TF' 'MT-TK'
 'MT-TL2' 'MT-TL1' 'MT-TI' 'MT-TH' 'MT-CO1;MT-TQ' 'MT-TE' 'MT-TN' 'MT-TA'
 'MT-ATP6;MT-ATP8'
 'covers 26 genes, none of which curated to show dosage sensitivity'
 'MT-ATP6' 'MT-CO3' 'MT-CO2' 'MT-CO1' 'MT-CYB' 'MT-ND6' 'MT-ND5;MT-ND6'
 'MT-ND5' 'MT-ND4L' 'MT-ND4' 'MT-ND3' 'MT-ND2' 'MT-ND1'
 'covers 22 genes, none of which curated to show dosage sensitivity' 'RS1'
 'CDKL5;RS1' 'RPGR' 'IL2RG' 'F8' 'F9' 'PDHA1' 'MTM1' 'CDKL5' 'SLC6A8'
 'MECP2' 'MYO7A' 'IDUA' 'IDUA;SLC26A1' 'CDH1' 'TP53' 'TNNT2' 'TNNI3'
 '

In [14]:
print(clinvar_f["ReferenceAlleleVCF"].unique())

['A' 'G' 'C' 'T']


In [15]:
clinvar_f.to_csv(f'{INTERIM}/clinvar_1.csv')